In [10]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor
import boto3

from model import ViTYOLO

In [2]:
ds = load_dataset("rishitdagli/cppe-5")

ds, ds["train"][0]

(DatasetDict({
     train: Dataset({
         features: ['image_id', 'image', 'width', 'height', 'objects'],
         num_rows: 1000
     })
     test: Dataset({
         features: ['image_id', 'image', 'width', 'height', 'objects'],
         num_rows: 29
     })
 }),
 {'image_id': 15,
  'image': <PIL.Image.Image image mode=RGB size=943x663>,
  'width': 943,
  'height': 663,
  'objects': {'id': [114, 115, 116, 117],
   'area': [3796, 1596, 152768, 81002],
   'bbox': [[302.0, 109.0, 73.0, 52.0],
    [810.0, 100.0, 57.0, 28.0],
    [160.0, 31.0, 248.0, 616.0],
    [741.0, 68.0, 202.0, 401.0]],
   'category': [4, 4, 0, 0]}})

In [3]:
train, test = ds["train"], ds["test"]

features = train.features
categories = features["objects"].feature["category"].names

labels = {0: "None"}

for i, category in enumerate(categories):
    labels[i + 1] = category

num_labels = len(categories)

labels, num_labels

({0: 'None',
  1: 'Coverall',
  2: 'Face_Shield',
  3: 'Gloves',
  4: 'Goggles',
  5: 'Mask'},
 5)

In [4]:
checkpoint = "google/vit-base-patch16-224-in21k"

num_classes = num_labels
boxes_per_cell = 3
grid_size = 7
hidden_size = 2048

processor = AutoImageProcessor.from_pretrained(checkpoint)
model = ViTYOLO(num_classes, boxes_per_cell, grid_size, hidden_size) # add one class for the zero input

In [8]:
def preprocess_image(record, processor):
    processed = processor(images=record["image"], return_tensors="pt")

    return processed["pixel_values"]

def process_for_training(records, processor, num_classes, boxes_per_cell, grid_size):
    x_out = []
    y_out = []

    for record in records:
        try:
            # Calculate X
            tmp_x = preprocess_image(record, processor)

            # Calculate Y
            width = record["width"]
            height = record["height"]

            tmp_y = [[] for _ in range(grid_size * grid_size)]
            bbox = record["objects"]["bbox"]
            categories = record["objects"]["category"]

            cell_width = width // grid_size
            cell_height = height // grid_size

            for i, (x, y, dx, dy) in enumerate(bbox):
                center_x = (x + dx) / 2
                center_y = (y + dy) / 2

                # Calculate the result
                res = [0 for _ in range(4 + num_classes + 1)]
                res[0] = x / width
                res[1] = y / height
                res[2] = dx / width
                res[3] = dy / height

                res[4] = 1 # Set the confidence score

                res[4 + categories[i] + 1] = 1

                # Store the result
                cell_x = int(center_x / cell_width)
                cell_y = int(center_y / cell_height)

                idx = cell_y * grid_size + cell_x

                tmp_y[idx].append(res)

            # Ensure the number of boxes per cell are equal
            for i in range(len(tmp_y)):
                if len(tmp_y[i]) > boxes_per_cell:
                    tmp_y[i] = tmp_y[i][:boxes_per_cell]

                # Pad the results with empty boxes to meet box requirement
                diff = max(0, boxes_per_cell - len(tmp_y[i]))
                for _ in range(diff):
                    tmp_y[i].append([0 for _ in range(4 + num_classes + 1)])

            x_out.append(tmp_x)
            y_out.append(tmp_y)

        except Exception as e:
            print(f"exception {e} for record {record}, skipping")

    return torch.stack(x_out), torch.tensor(y_out).view(-1, grid_size * grid_size * boxes_per_cell * (4 + num_classes + 1))

x_train, y_train = process_for_training(ds["train"], processor, num_classes, boxes_per_cell, grid_size)
x_test, y_test = process_for_training(ds["test"], processor, num_classes, boxes_per_cell, grid_size)

exception Unable to infer channel dimension format for record {'image_id': 218, 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=375x500 at 0x141BE2D80>, 'width': 54, 'height': 215, 'objects': {'id': [1374, 1375], 'area': [2584, 102090], 'bbox': [[159.0, 73.0, 68.0, 38.0], [77.0, 2.0, 205.0, 498.0]], 'category': [4, 0]}}, skipping
exception Unable to infer channel dimension format for record {'image_id': 266, 'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=484x500 at 0x141BE1370>, 'width': 295, 'height': 500, 'objects': {'id': [1545, 1546, 1547, 1548, 1549], 'area': [7047, 1768, 5175, 3087, 78490], 'bbox': [[199.0, 22.0, 81.0, 87.0], [215.0, 58.0, 52.0, 34.0], [164.0, 262.0, 69.0, 75.0], [280.0, 264.0, 49.0, 63.0], [164.0, 6.0, 167.0, 470.0]], 'category': [1, 4, 2, 2, 0]}}, skipping
exception Unable to infer channel dimension format for record {'image_id': 144, 'image': <PIL.Image.Image image mode=CMYK size=800x451 at 0x136951460>, 'width': 800, 'height': 4

In [9]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

(torch.Size([963, 1, 3, 224, 224]),
 torch.Size([963, 1470]),
 torch.Size([28, 1, 3, 224, 224]),
 torch.Size([28, 1470]))

In [6]:
inputs = x_test[0]

with torch.no_grad():
    outputs = model(inputs)

inputs.shape, outputs, outputs.shape

(torch.Size([1, 3, 224, 224]),
 tensor([[-0.0100,  0.0302, -0.0101,  ...,  0.0194,  0.0470, -0.0264]]),
 torch.Size([1, 1470]))

In [16]:
region = "ap-southeast-2"

s3_client = boto3.client("s3", region_name=region)

bucket_prefix = "vision-demo"
bucket_name = "sagemaker-ap-southeast-2-879381285437"

bucket_prefix, region, bucket_name

('vision-demo', 'ap-southeast-2', 'sagemaker-ap-southeast-2-879381285437')

In [20]:
x_train_file = "x_train.pt"
y_train_file = "y_train.pt"
x_test_file = "x_test.pt"
y_test_file = "y_test.pt"

torch.save(x_train, x_train_file)
torch.save(y_train, y_train_file)
torch.save(x_test, x_test_file)
torch.save(y_test, y_test_file)

data_path = f"{bucket_prefix}/data"

s3_client.upload_file(x_train_file, bucket_name, f"{data_path}/{x_train_file}")
s3_client.upload_file(y_train_file, bucket_name, f"{data_path}/{y_train_file}")
s3_client.upload_file(x_test_file, bucket_name, f"{data_path}/{x_test_file}")
s3_client.upload_file(y_test_file, bucket_name, f"{data_path}/{y_test_file}")